# Inspect Stage-1 Bundle
This notebook loads `artifacts/stage1_production_bundle.pkl` and `artifacts/stage1_feature_mapping.json` and produces a human-friendly report of the feature ordering, model metadata, and intersections with `uav_risk.ml.feature_defs`. Use this to decide permanent fixes to artifacts vs code.

In [ ]:
import joblib, json, os
from pathlib import Path
repo_root = Path('.').resolve()
art = repo_root / 'artifacts'
bundle_path = art / 'stage1_production_bundle.pkl'
mapping_path = art / 'stage1_feature_mapping.json'
print('bundle exists?', bundle_path.exists())
print('mapping exists?', mapping_path.exists())
bundle = joblib.load(bundle_path)
with open(mapping_path, 'r', encoding='utf-8') as f:
    mapping = json.load(f)
fn = bundle.get('feature_names')
print('bundle.feature_names: count=', len(fn))
print('first 20:', fn[:20])
print('last 5:', fn[-5:])
print('bundle keys:', list(bundle.keys()))
print('mapping total_features:', mapping.get('total_features'))
print('mapping first20:', mapping.get('feature_names')[:20])
print('class_names in artifact mapping:', mapping.get('class_names'))
print('model card sample metadata from bundle (if present):')
print('bundle.metadata' if 'metadata' in bundle else 'no metadata in bundle')

In [ ]:
# Cross-reference with feature_defs
from uav_risk.ml.feature_defs import get_all_feature_definitions, get_all_feature_names, get_safe_value
import pandas as pd
code_defs = get_all_feature_definitions()
art_list = mapping.get('feature_names') if isinstance(mapping, dict) else mapping
rows = []
for i, name in enumerate(art_list):
    defn = code_defs.get(name)
    rows.append({
        'index': i,
        'feature_name': name,
        'in_code_defs': name in code_defs,
        'safe_value': get_safe_value(name),
        'is_core_in_code': (defn.get('is_core') if defn else False),
    })
df = pd.DataFrame(rows)
pd.set_option('display.max_rows', 300)
df.head(40)

In [ ]:
# Save a CSV report for convenience
out = repo_root / 'artifacts' / 'stage1_feature_report.csv'
df.to_csv(out, index=False)
print('Wrote', out)